In [ ]:
import pandas as pd

df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [ ]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('.', '  ')
  text = text.replace("'", '  ')
  text = text.replace('?', '  ')
  text = text.replace('!', '  ')
  return text.split()

In [ ]:
tokenize('What about u?')

['what', 'about', 'u']

In [ ]:
# vocab
vocab = {'<UNK>':0}

In [ ]:
def build_vocab(row):
  question = tokenize(row['question'])
  answer = tokenize(row['answer'])

  merge_tokens = question + answer

  for token in merge_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)


In [ ]:
# convert word to numerical indiecs
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [ ]:
len(vocab)

324

In [ ]:
# convert words to numerical indices.
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text


In [ ]:
text_to_indices('What is campuX', vocab)

[1, 2, 0]

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    numerical_question = text_to_indices(self.df.iloc[idx]['question'], vocab)
    numerical_answer = text_to_indices(self.df.iloc[idx]['answer'], vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [ ]:
dataset = QADataset(df, vocab)

In [ ]:
dataset[10]

(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
for question, answer in dataloader:
  print(question)
  print(answer)


tensor([[ 10,  11, 157, 158, 159]])
tensor([[160]])
tensor([[ 42, 167,   2,   3,  17, 168, 169]])
tensor([[170]])
tensor([[  1,   2,   3,  69,   5, 155]])
tensor([[156]])
tensor([[  1,   2,   3,   4,   5, 236, 237]])
tensor([[238]])
tensor([[10, 55,  3, 56,  5, 57]])
tensor([[58]])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]])
tensor([[173]])
tensor([[ 42, 250, 251, 118, 252, 253]])
tensor([[254]])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]])
tensor([[128]])
tensor([[ 42, 299, 300, 118,  14, 301, 302, 158, 303, 304, 305, 306]])
tensor([[307]])
tensor([[ 1,  2,  3, 33, 34,  5, 35]])
tensor([[36]])
tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]])
tensor([[28]])
tensor([[  1,   2,   3,   4,   5, 206]])
tensor([[207]])
tensor([[ 1,  2,  3,  4,  5, 53]])
tensor([[54]])
tensor([[42, 18,  2, 62, 63,  3, 64, 18]])
tensor([[65]])
tensor([[  1,   2,   3,  33,  34,   5, 245]])
tensor([[246]])
tensor([[ 42, 137,   2, 138,  39, 175, 269]])
tensor([[99]])
tensor([[ 1,  2,  3, 92, 93, 9

# Architechture

In [ ]:
import torch
import torch.nn as nn

In [ ]:
dataset[15][0]

tensor([ 1,  2,  3, 69,  5,  3, 70, 71])

In [ ]:
dataset[15][0].shape

torch.Size([8])

In [ ]:
len(vocab)

324

In [ ]:
x = nn.Embedding(num_embeddings=len(vocab), embedding_dim=50)

In [ ]:
a = x(dataset[15][0])

In [ ]:
a.shape

torch.Size([8, 50])

In [ ]:
y = nn.RNN(input_size=a.shape[1], hidden_size=64)

In [ ]:
y(a)[1].shape

torch.Size([1, 64])

In [ ]:
y(a)[0].shape

torch.Size([8, 64])

In [ ]:
# final output
b = y(a)[1]

In [ ]:
b.shape[1]

64

In [ ]:
z = nn.Linear(in_features=b.shape[1], out_features=len(vocab))

In [ ]:
z(b).shape

torch.Size([1, 324])

In [ ]:
class simpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(input_size=50, hidden_size=64, batch_first=True)
    self.linear = nn.Linear(in_features=64, out_features=vocab_size)

  def forward(self, question):
    embedded_ques = self.embedding(question)
    hidden, final = self.rnn(embedded_ques)
    output = self.linear(final.squeeze(0))

    return output





In [ ]:
learning_rate = 0.001
epochs = 20

In [ ]:
model = simpleRNN(vocab_size=len(vocab))

In [ ]:
criteria = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:
    #for clearing the gradiant
    optimizer.zero_grad()

    # forward prop
    output = model(question)

    # loss - squeeze the target to match expected 1D shape [batch_size]
    loss = criteria(output, answer.squeeze(-1))

    # gradiant
    loss.backward()

    # update
    optimizer.step()

    total_loss += loss.item()

  print(f'Epoch: {epoch+1} Loss: {total_loss}')

Epoch: 1 Loss: 519.7911367416382
Epoch: 2 Loss: 453.1641068458557
Epoch: 3 Loss: 376.8197078704834
Epoch: 4 Loss: 313.2338147163391
Epoch: 5 Loss: 261.38822984695435
Epoch: 6 Loss: 214.62904262542725
Epoch: 7 Loss: 171.65070009231567
Epoch: 8 Loss: 134.77999705076218
Epoch: 9 Loss: 104.14265066385269
Epoch: 10 Loss: 80.04476642608643
Epoch: 11 Loss: 61.97951924800873
Epoch: 12 Loss: 48.94771145284176
Epoch: 13 Loss: 39.537392035126686
Epoch: 14 Loss: 31.877785205841064
Epoch: 15 Loss: 26.302422389388084
Epoch: 16 Loss: 21.91892833262682
Epoch: 17 Loss: 18.604513190686703
Epoch: 18 Loss: 15.870164416730404
Epoch: 19 Loss: 13.67301121354103
Epoch: 20 Loss: 11.893428537994623


In [ ]:
def predict(model, question):
  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert to logoc
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob

  value, index = torch.max(probs, dim=1)

  print(value, index)

  if value < 0.5:
    print('I donot know')

  else:
    ans =list(vocab.keys())[index]
    print(ans)


In [ ]:
predict(model, 'France?')

tensor([0.6272], grad_fn=<MaxBackward0>) tensor([7])
paris
